# 06 - Evaluation: Statistik Komposisi & Proyeksi WMA

## Tujuan
Menguji apakah komposisi topik berubah signifikan antar tahun, lalu memproyeksikan
tren distribusi topik 2026-2027 dengan Weighted Moving Average.

## Asal (Input)
- `_fix_final/topic_by_year_final.csv` (dari Notebook 08)
- `_fix_final/topic_by_year_sensitivity.csv`
- `_label_work/labels.json`

## Keluaran (Output)
- `_stats_forecast/forecast_table.csv` - proyeksi per topik
- `_stats_forecast/backtest_results.csv` - MAE walk-forward
- `_stats_forecast/share_intervals.csv` - interval Wilson 95%
- `_stats_forecast/per_topic_tests.csv` - uji Spearman + Bonferroni
- `_stats_forecast/composition_tests_report.md` - chi-square
- `_stats_forecast/sensitivity_report.md` - uji sensitivitas

## Metodologi Proyeksi
WMA dengan bobot linear [1,2,3,4,5] diterapkan pada 5 titik historis (2021-2025).
Proyeksi 2026 dihitung dari data asli; 2027 bersifat rekursif (indikatif).
Karena hanya tersedia 5 titik, seluruh proyeksi **tidak dapat diklaim
sebagai prediksi statistik yang andal**.

In [1]:
import sys, io as _io
from pathlib import Path
_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(_ROOT))
try:
    sys.stdout.reconfigure(encoding='utf-8', errors='replace')
except Exception:
    pass

import pandas as pd, numpy as np
from src import config as C
C.ensure_dirs()
print(C.describe())

KONFIGURASI PROYEK
  Repo          : C:\Users\HAJIWAHYU\Downloads\lda-topik-skripsi-si-uinrf-main
  Dokumen akhir : 285 (2021, 2022, 2023, 2024, 2025)
  Vocabulary    : 1849
  Model LDA     : K=8, alpha=auto, eta=auto, passes=30, iterations=500, seed=42
  Inferensi     : Online Variational Bayes (bukan Gibbs Sampling)
  C_V resmi     : 0.4256
  Cutoff data   : 19 September 2026

## Langkah 1 - Jalankan Analisis Statistik & Proyeksi

In [2]:
from src import forecast
forecast.main()

=== Menjalankan Analisis Statistik & Proyeksi Tren Topik ===
 Saved composition_tests_report.md
 Saved per_topic_tests.csv & share_intervals.csv
 Saved backtest_results.csv & backtest_summary.md
 Saved forecast_table.csv
 Saved sensitivity_report.md
 Saved EVAL_NOTES.md
=== Seluruh Tugas Analisis Statistik Berhasil Dijalankan! ===

## Langkah 2 - Hasil Uji Komposisi (Chi-Square)

In [3]:
print(open(str(C.COMPOSITION_REPORT), encoding='utf-8').read()[:800])

# Laporan Uji Komposisi Topik x Tahun (2021-2025)

## 1. Ringkasan Statistik Contingency Table
- **Total Dokumen ($N$)**: 285
- **Dimensi Tabel**: 5 Tahun x 8 Topik (40 sel)
- **Derajat Kebebasan ($dof$)**: 28

## 2. Hasil Uji Chi-Square Homogenitas
- **Statistik $\chi^2$**: 23.9361
- **p-value Mentah (Asimptotik)**: 0.6849
- **Cramer's V**: 0.1449 (Mengindikasikan asosiasi sangat lemah hingga sedang)

## 3. Uji Permutasi Monte Carlo (10.000 iterasi, seed=42)
- **p-value Permutasi**: 0.6994
- **Kesimpulan Uji Homogenitas**: Tidak terdapat bukti statistik yang cukup untuk menyatakan komposisi topik berubah secara signifikan antar tahun (p >= 0.05)

## 4. Evaluasi Asumsi Nilai Harapan (Expected Frequencies)
- **Jumlah Sel dengan Nilai Harapan < 5**: 12 dari 40 sel (30.0%)
- **Peringatan Asum

## Langkah 3 - Tabel Proyeksi WMA

In [4]:
fc = pd.read_csv(C.FORECAST_CSV)
cols = ['topik_id','label','share_2025','wma_2026','pita_bawah_2026',
        'pita_atas_2026','wma_2027','arah']
print(fc[cols].to_string(index=False))

 topik_id                                                    label  share_2025  wma_2026  pita_bawah_2026  pita_atas_2026  wma_2027                   arah
        1                        Audit TI & Evaluasi Kesuksesan SI        7.06     11.39             3.37           19.41     11.10 tidak berubah bermakna
        2                 Pengembangan & Analisis Sistem Heterogen        8.24     10.05             2.70           17.39      9.87 tidak berubah bermakna
        3                    Analisis Kualitas Layanan & Risiko SI       17.65     15.19            10.54           19.83     15.29 tidak berubah bermakna
        4                     Kepuasan Pengguna Akhir (EUCS & TAM)       14.12     13.59             8.59           18.59     13.85 tidak berubah bermakna
        5                      Manajemen Risiko TI & Analitik Data       12.94     13.43             8.48           18.39     13.06 tidak berubah bermakna
        6                  Pengembangan & Perencanaan Strategis SI    

## Langkah 4 - Visualisasi Proyeksi

In [5]:
import plotly.graph_objects as go

tby = pd.read_csv(C.TOPIC_BY_YEAR_FINAL)
fig = go.Figure()
for t in [3, 4, 5, 6]:
    fig.add_trace(go.Scatter(
        x=['2021','2022','2023','2024','2025'],
        y=tby[f'proporsi_topik_{t}'],
        mode='lines+markers', name=f'Topik {t}'))
    row = fc[fc['topik_id']==t].iloc[0]
    fig.add_trace(go.Scatter(
        x=['2025','2026','2027'],
        y=[tby[f'proporsi_topik_{t}'].iloc[-1], row['wma_2026'], row['wma_2027']],
        mode='lines+markers', line=dict(dash='dot'),
        name=f'Topik {t} proyeksi', showlegend=False))

fig.add_vline(x=4, line_dash='dash', line_color='#999')
fig.update_layout(template='plotly_white', height=460,
                  xaxis_title='Tahun', yaxis_title='Proporsi (%)')
fig.show()

## Langkah 5 - Hasil Walk-Forward Backtest

In [6]:
bt = pd.read_csv(C.BACKTEST_CSV)
wma = bt[bt['metode']=='WMA']
hist= bt[bt['metode']=='Rata-rata Historis']
print(f'MAE rerata WMA            : {wma["mae_pp"].mean():.2f} pp')
print(f'MAE rerata Rata-rata Hist.: {hist["mae_pp"].mean():.2f} pp')
print()
better = (wma['mae_pp'].values < hist['mae_pp'].values).sum()
print(f'WMA lebih akurat pada {better} dari 8 topik')
print()
print('Kesimpulan: WMA TIDAK secara konsisten mengalahkan baseline rerata,')
print('sehingga seluruh proyeksi hanya bersifat indikatif.')

MAE rerata WMA            : 4.46 pp
MAE rerata Rata-rata Hist.: 4.35 pp

WMA lebih akurat pada 2 dari 8 topik

Kesimpulan: WMA TIDAK secara konsisten mengalahkan baseline rerata,
sehingga seluruh proyeksi hanya bersifat indikatif.

## Ringkasan Fase 06 - Evaluation & Forecast

| Temuan | Nilai |
|:---|---:|
| Chi-square p-value | 0,6861 (tidak signifikan) |
| Cramér's V | 0,1448 (asosiasi lemah) |
| MAE rerata WMA | 4,64 pp |
| Topik dengan WMA lebih baik | 2 dari 8 |

Tidak ditemukan pergeseran komposisi topik yang signifikan secara statistis
selama 2021-2025. Proyeksi WMA hanya indikatif dan tidak boleh diklaim
sebagai prediksi yang andal.

## Peta Artefak

| Artefak | Dibuat oleh | Consumed by |
|:---|:---|:---|
| `_stats_forecast/forecast_table.csv` | Notebook 06 | app (halaman Proyeksi) |
| `_stats_forecast/backtest_results.csv` | Notebook 06 | Lampiran Bab IV |
| `_stats_forecast/share_intervals.csv` | Notebook 06 | app (halaman Tren) |